[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

Реализуйте **Layer Normalization** с нуля. Для каждого отдельного объекта LayerNorm нормализует признаки последней dimension независимо от других объектов batch. Поэтому результат одного token не зависит от соседних token или размера batch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

где $\mu$ и $\sigma^2$ вычисляются по **последней dimension** с `unbiased=False`. `eps` предотвращает деление на ноль, а `gamma` и `beta` выполняют обучаемое поэлементное affine-преобразование.

### Формы и broadcasting
Для входа `(B, T, D)` mean и variance удобно хранить как `(B, T, 1)` через `keepdim=True`. Тогда стандартизированный tensor снова имеет `(B, T, D)`, а `gamma` и `beta` формы `(D,)` автоматически broadcast по `B` и `T`.

### LayerNorm и BatchNorm
- LayerNorm редуцирует последнюю feature-axis внутри каждого объекта;
- BatchNorm редуцирует batch-axis для каждой feature;
- LayerNorm использует текущие statistics и в train, и в eval, поэтому running statistics не нужны.

### Контракт
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### План реализации
1. Вычислите mean по `dim=-1` с сохранением dimension.
2. Вычислите biased variance как mean квадратов отклонений.
3. Стандартизируйте вход через `sqrt(var + eps)`.
4. Примените `gamma` и `beta` с broadcasting.

### Ограничения
- не используйте `F.layer_norm` или `torch.nn.LayerNorm`;
- нормализуйте только последнюю dimension;
- сохраняйте autograd для `x`, `gamma`, `beta`.

### Быстрые самопроверки
- при `gamma=1`, `beta=0` mean последней dimension около нуля, variance около единицы;
- формы `(B,D)` и `(B,T,D)` сохраняются;
- изменение одного объекта batch не влияет на остальные;
- результат совпадает с `F.layer_norm`;
- backward заполняет gradients входа и affine parameters.

### Типичные ошибки
- reduction по `dim=0`;
- `unbiased=True`, не совпадающий с контрактом;
- потерянная dimension без `keepdim`;
- `eps` добавляется после square root;
- gamma и beta применяются до стандартизации.

### Официальные материалы и примеры
- [`torch.nn.LayerNorm`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html) — formula, axes, affine parameters и NLP-пример;
- [`torch.var`](https://docs.pytorch.org/docs/stable/generated/torch.var.html) — параметры correction и keepdim.

### Вопросы для самопроверки
1. Почему LayerNorm не требует running statistics?
2. Какая shape будет у mean для входа `(2,4,8)` при `keepdim=True`?
3. Зачем после нормализации нужны gamma и beta?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_layer_norm(x, gamma, beta, eps=1e-5):
    pass  # Replace this

In [ ]:
# 🧪 Debug
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("layernorm")